***1. Import libraries***

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import folium
from datetime import datetime
import geopandas as gpd
from shapely.geometry import Point, box
from IPython.display import IFrame
from scipy.interpolate import PchipInterpolator


***2. Load Your Dataset***
***You change only the file name below every day***

In [ ]:
file_path = r"C:\Users\Jeen priyee\Downloads\CEEW-Daily report_GDA-open-burning\complaint-1oct-31dec.xlsx" # CHANGE THIS ONLY
df = pd.read_excel(file_path)

df.head()


In [ ]:
print(df.columns.tolist())

In [ ]:
df['Offences'].unique()


***4. Main Summary Numbers***

In [ ]:
# Total complaints
total_complaints = df['Compliant ID'].nunique()

# Count of Resolved complaints
resolved_count = df[df['Status'] == 'Resolved']['Compliant ID'].nunique()

# Count of Pending complaints
pending_count = df[df['Status'] == 'Pending']['Compliant ID'].nunique()
rejected_count = df[df['Status'] == 'Rejected']['Compliant ID'].nunique()
unrelated_count = df[df['Status'] == 'Unrelated']['Compliant ID'].nunique()

print("Total Complaints:", total_complaints)
print("Resolved Complaints:", resolved_count)
print("Pending Complaints:", pending_count)
print("Rejected Complaints:", rejected_count)
print("Unrelated Complaints:", unrelated_count)


***5. Filter Burning Complaints (Offence 74 & 113)***

In [ ]:
# Filter burning-related complaints

burning_df = df[df["Offences"].isin([74, 113])]
burning_df = burning_df.drop_duplicates(subset="Compliant ID")
burning_count = burning_df['Compliant ID'].nunique()


# Percentage
burning_percentage = (burning_count / total_complaints) * 100

# Pending cases
burning_pending = burning_df[burning_df['Status'] == 'Pending']['Compliant ID'].nunique()

# Resolved cases
burning_resolved = burning_df[burning_df['Status'] == 'Resolved']['Compliant ID'].nunique()
burning_rejected = burning_df[burning_df['Status'] == 'Rejected']['Compliant ID'].nunique()
burning_unrelated = burning_df[burning_df['Status'] == 'Unrelated']['Compliant ID'].nunique()


print("\n--- OPEN BURNING-RELATED COMPLAINTS SUMMARY ---")
print("Burning-Related Complaints:", burning_count)
print(f"Percentage of Burning Complaints: {burning_percentage:.2f}%")
print("Burning Resolved:", burning_resolved)
print("Burning Pending:", burning_pending)
print("Burning Rejected:", burning_rejected)
print("Burning Unrelated:", burning_unrelated)


***6. Burning Complaints details***

In [ ]:
burning_details = burning_df[['Compliant ID', 'District', 'Status', 'Date and Time', 'Date & Time of Resolution']]

print(burning_details.to_string())

***7. District-wise Burning Distribution***

In [ ]:
district_burning = burning_df['District'].value_counts()
district_burning

***8. SLA Performance***

In [ ]:
burn = burning_df.copy()

# Convert datetime columns
burn["Date and Time"] = pd.to_datetime(burn["Date and Time"], errors="coerce")
burn["Date & Time of Resolution"] = pd.to_datetime(burn["Date & Time of Resolution"], errors="coerce")

# 1. Compute Resolution Time (hours)
burn["Resolution_Time_hours"] = (
    (burn["Date & Time of Resolution"] - burn["Date and Time"])
    .dt.total_seconds() / 3600
)

# 2. SLA = 2 hours
SLA_hours = 2

# 3. SLA Classification
def within_sla(row):
    if pd.isna(row["Resolution_Time_hours"]):
        return "Pending"
    return "Yes" if row["Resolution_Time_hours"] <= SLA_hours else "No"

burn["Within_SLA"] = burn.apply(within_sla, axis=1)

# 4. Table with required columns
import pandas as pd
sla_table = burn[['Compliant ID','District','Status','Date and Time',
                  'Date & Time of Resolution','Resolution_Time_hours','Within_SLA']].reset_index(drop=True)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 2000)

print(sla_table)

# 5. Percentage Summary
total = len(burn)
within = (burn["Within_SLA"] == "Yes").sum()
after = (burn["Within_SLA"] == "No").sum()
pending = (burn["Within_SLA"] == "Pending").sum()

print("\n--- SLA PERFORMANCE SUMMARY ---")
print(f"Total Burning Complaints: {total}")
print(f"Resolved Within SLA: {within}  ({within/total*100:.2f}%)")
print(f"Resolved/Pending After SLA: {after+pending}  ({(after+pending)/total*100:.2f}%)")


# ------------------------------------------------------------
# 6. SPEED OF WORK (existing section)
# ------------------------------------------------------------

resolved = burn[burn["Resolution_Time_hours"].notna()]

if len(resolved) > 0:
    avg_speed = resolved["Resolution_Time_hours"].mean()
    median_speed = resolved["Resolution_Time_hours"].median()
    fastest = resolved["Resolution_Time_hours"].min()
    slowest = resolved["Resolution_Time_hours"].max()

    print("\n--- Resolution Speed ---")
    print(f"Average Resolution Time: {avg_speed:.2f} hours")
    print(f"Median Resolution Time: {median_speed:.2f} hours")
    print(f"Fastest Resolution: {fastest:.2f} hours")
    print(f"Slowest Resolution: {slowest:.2f} hours")
else:
    print("\nNo resolved complaints found → cannot compute speed metrics.")


# ------------------------------------------------------------
# 7. NEW (ONLY NECESSARY ADDITION): 
#     Longest Pending + Slowest Resolved Complaints
# ------------------------------------------------------------

# A. Compute time open for pending complaints
now = pd.Timestamp.now()
burn["Time_Open_hours"] = (
    (now - burn["Date and Time"]).dt.total_seconds() / 3600
)

# B. Longest Pending Complaint
pending_df = burn[burn["Resolution_Time_hours"].isna()]

if len(pending_df) > 0:
    longest_pending = pending_df.loc[pending_df["Time_Open_hours"].idxmax()]

    print("\n--- Longest Pending Complaint ---")
    print(longest_pending[["Compliant ID", "Status", "Date and Time", "Time_Open_hours"]])
else:
    print("\nNo pending complaints found.")

# C. Slowest Resolved Complaint (Full details)
if len(resolved) > 0:
    slowest_resolved = resolved.loc[resolved["Resolution_Time_hours"].idxmax()]

    print("\n--- Slowest Resolved Complaint (Detailed) ---")
    print(slowest_resolved[["Compliant ID", "Status", "Date and Time",
                            "Date & Time of Resolution", "Resolution_Time_hours"]])
else:
    print("\nNo resolved complaints found.")


In [ ]:
# SLA category counts
sla_counts = burn["Within_SLA"].value_counts()

# Ensure all categories exist (for consistent ordering)
sla_counts = sla_counts.reindex(["Yes", "No", "Pending"], fill_value=0)

labels = [
    "Resolved within SLA",
    "Resolved beyond SLA",
    "Pending"
]
sizes = sla_counts.values

# Balanced (not too light, not too dark) color palette
colors = ["#8cb731", "#029bd6", "#e85724"]  # green, blue, red (muted)

plt.figure(figsize=(5, 5))

plt.pie(
    sizes,
    labels=labels,
    colors=colors,
    autopct='%1.1f%%',
    startangle=90
)

plt.title("SLA Compliance Status – Open Burning Complaints")
plt.axis("equal")
plt.tight_layout()
plt.show()



In [ ]:
plt.figure(figsize=(8, 4))

plt.hist(
    resolved["Resolution_Time_hours"],
    bins=20
)

plt.xlabel("Resolution Time (hours)")
plt.ylabel("Number of Complaints")
plt.title("Distribution of Resolution Time for Open Burning Complaints")

plt.tight_layout()
plt.show()


***9. Peak Time for burning related complaints graph***

In [ ]:
# Convert datetime
burn["Date and Time"] = pd.to_datetime(burn["Date and Time"], errors="coerce")

# Create hourly complaint counts
burn["Hour"] = burn["Date and Time"].dt.hour
hourly_counts = burn["Hour"].value_counts().sort_index()

# Prepare data for smoothing
hours = hourly_counts.index.values
counts = hourly_counts.values

x_new = np.linspace(hours.min(), hours.max(), 300)
pchip = PchipInterpolator(hours, counts)
y_new = pchip(x_new)

# Plot Smooth Curve
plt.figure(figsize=(8,4))

# Smooth PCHIP curve
plt.plot(x_new, y_new, linewidth=3, color="#029bd6")

# Original scatter points
plt.scatter(hours, counts, color="#e85724", s=60)

plt.title("Peak Time for Burning-Related Complaints", fontsize=18)
plt.xlabel("Hour of Day (0–23)", fontsize=14)
plt.ylabel("Number of Complaints", fontsize=14)

plt.xticks(range(0, 24))
plt.yticks(np.arange(0, counts.max() + 2, 4))

plt.grid(True, linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()


***10. Maps for Burning related complaints (Pending = Red, Resolved = green)***

In [ ]:
# =====================================================
# COMPLETE END-TO-END SCRIPT
# Burning-related complaints: Ward | Count | Avg SLA
# (Avg SLA calculated on resolved complaints only)
# =====================================================

import pandas as pd
import geopandas as gpd
from IPython.display import display

# =========================
# 1. Load complaints data
# =========================
file_path = r"C:\Users\Jeen priyee\Downloads\CEEW-Daily report_GDA-open-burning\complaint-1oct-31dec.xlsx"
df = pd.read_excel(file_path)

# =========================
# 2. Fix mojibake (Hindi text)
# =========================
def fix_mojibake(x):
    if isinstance(x, str):
        try:
            return x.encode("latin1").decode("utf-8")
        except:
            return x
    return x

df = df.map(fix_mojibake)

# =========================
# 3. Split Latitude & Longitude
# =========================
df[['Latitude', 'Longitude']] = df['Latitude & Longitude'].str.split(",", expand=True)
df['Latitude'] = df['Latitude'].astype(float)
df['Longitude'] = df['Longitude'].astype(float)

# =========================
# 4. Filter Burning Complaints
# =========================
burn = df[df["Offences"].isin([74, 113])].copy()
print("Total Burning Complaints:", len(burn))

# =========================
# 5. Datetime conversion
# =========================
burn["Date and Time"] = pd.to_datetime(burn["Date and Time"], errors="coerce")
burn["Date & Time of Resolution"] = pd.to_datetime(
    burn["Date & Time of Resolution"], errors="coerce"
)

# =========================
# 6. SLA computation
# =========================
burn["Resolution_Time_hours"] = (
    (burn["Date & Time of Resolution"] - burn["Date and Time"])
    .dt.total_seconds() / 3600
)

SLA_hours = 2

burn["Within_SLA"] = burn["Resolution_Time_hours"].apply(
    lambda x: "Pending" if pd.isna(x) else ("Yes" if x <= SLA_hours else "No")
)

# =========================
# 7. Load Delhi wards shapefile
# =========================
shp_path = r"C:\Users\Jeen priyee\Downloads\Delhi_Wards-SHP_Datameet\Delhi_Wards.shp"
delhi_gdf = gpd.read_file(shp_path).to_crs("EPSG:4326")

# =========================
# 8. Convert to GeoDataFrame
# =========================
burning_gdf = gpd.GeoDataFrame(
    burn,
    geometry=gpd.points_from_xy(burn["Longitude"], burn["Latitude"]),
    crs="EPSG:4326",
)

# =========================
# 9. Spatial join (Assign ward)
# =========================
burning_with_ward = gpd.sjoin(
    burning_gdf,
    delhi_gdf[["Ward_Name", "geometry"]],
    how="left",
    predicate="within",
)

# =========================
# 10. FINAL TABLE
#     Ward | Count | Avg SLA (resolved only)
# =========================
ward_sla_table = (
    burning_with_ward
    .groupby("Ward_Name")
    .agg(
        Complaint_Count=("Ward_Name", "size"),
        Avg_SLA_hours=("Resolution_Time_hours", lambda x: x.dropna().mean())
    )
    .reset_index()
    .sort_values(by="Complaint_Count", ascending=False)
)

# Round SLA
ward_sla_table["Avg_SLA_hours"] = ward_sla_table["Avg_SLA_hours"].round(2)

# Replace NaN SLA with readable label
ward_sla_table["Avg_SLA_hours"] = ward_sla_table["Avg_SLA_hours"].fillna("Not Available")

# =========================
# 11. Display in Jupyter
# =========================
display(ward_sla_table)

# =========================
# 12. Save output
# =========================
output_path = r"C:\Users\Jeen priyee\Downloads\ward_burning_count_avg_sla.xlsx"
ward_sla_table.to_excel(output_path, index=False)

print("Saved table to:", output_path)


In [ ]:
# Ensure datetime format
burning_df["Date and Time"] = pd.to_datetime(
    burning_df["Date and Time"], errors="coerce"
)

# Keep only Oct–Dec
burning_df = burning_df[
    burning_df["Date and Time"].dt.month.isin([10, 11, 12])
]

# Create weekly period
burning_df["Week"] = burning_df["Date and Time"].dt.to_period("W")

# Weekly aggregation
weekly_counts = (
    burning_df
    .groupby("Week")
    .size()
    .reset_index(name="Number of Complaints")
)

# Sort by week
weekly_counts = weekly_counts.sort_values("Week")


In [ ]:
# Reset index to create sequential week numbers
weekly_counts = weekly_counts.reset_index(drop=True)

# Create Week 1, Week 2, ...
weekly_counts["Week_Label"] = [
    f"Week {i+1}" for i in range(len(weekly_counts))
]


In [ ]:
plt.figure(figsize=(10, 4))
plt.bar(
    weekly_counts["Week_Label"],
    weekly_counts["Number of Complaints"]
)
plt.xlabel("Weeks (Oct–Dec)")
plt.ylabel("Number of Complaints")
plt.title("Weekly Trend of Open Burning Complaints (Oct–Dec)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 4))
plt.bar(
    weekly_counts["Week_Label"],
    weekly_counts["Number of Complaints"],
    color="#029bd6"
)
plt.xlabel("Weeks (Oct–Dec)")
plt.ylabel("Number of Complaints")
plt.title("Weekly Trend of Open Burning Complaints (Oct–Dec)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 4))
plt.bar(
    weekly_counts["Week"].astype(str),
    weekly_counts["Number of Complaints"]
)
plt.xlabel("Weeks (Oct–Dec)")
plt.ylabel("Number of Complaints")
plt.title("Weekly Trend of Open Burning Complaints (Oct–Dec)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(
    weekly_counts["Week"].astype(str),
    weekly_counts["Number of Complaints"],
    marker="o",
    color="#029bd6"
)
plt.xlabel("Weeks (Oct–Dec)")
plt.ylabel("Number of Complaints")
plt.title("Weekly Trend of Open Burning Complaints (Oct–Dec)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
#Chloropleth map that shows complaint count of all the wards highlighting top 5 wards
import pandas as pd
import geopandas as gpd
import folium
from shapely.geometry import Point, box
from shapely.geometry.polygon import Polygon
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.patches as mpatches

# Load the CSV file with a specified encoding
csv_path = r"C:\Users\Jeen priyee\Downloads\CEEW-Daily report_GDA-open-burning\complaint-1oct-31dec.xlsx" # CHANGE THIS ONLY

complaints_df = pd.read_excel(csv_path)

# Split latitude and longitude into separate columns
complaints_df[['Latitude', 'Longitude']] = complaints_df['Latitude & Longitude'].str.split(',', expand=True)
complaints_df['Latitude'] = complaints_df['Latitude'].astype(float)
complaints_df['Longitude'] = complaints_df['Longitude'].astype(float)

# Use all complaints instead of filtering for pending ones
all_complaints_df = burning_df

# Load the shapefile
shp_path = r'C:\Users\Jeen priyee\Downloads\Delhi_Wards-SHP_Datameet\Delhi_Wards.shp'
delhi_gdf = gpd.read_file(shp_path)

# Create GeoDataFrame for all complaints
geometry = [Point(xy) for xy in zip(all_complaints_df['Longitude'], all_complaints_df['Latitude'])]
all_complaints_gdf = gpd.GeoDataFrame(all_complaints_df, geometry=geometry)

# Set CRS for both GeoDataFrames if needed
if delhi_gdf.crs is None:
    delhi_gdf.set_crs(epsg=4326, inplace=True)
    
all_complaints_gdf.crs = delhi_gdf.crs

# Perform a spatial join to aggregate occurrences by region
joined_gdf = gpd.sjoin(all_complaints_gdf, delhi_gdf, how='left', predicate='within')

# Count occurrences in each region
region_counts = joined_gdf.groupby('index_right').size()

# Get ward name information if available in the shapefile
if 'name' in delhi_gdf.columns:
    ward_name_col = 'name'
elif 'NAME' in delhi_gdf.columns:
    ward_name_col = 'NAME'
elif 'Ward_Name' in delhi_gdf.columns:
    ward_name_col = 'Ward_Name'
elif 'WARD_NAME' in delhi_gdf.columns:
    ward_name_col = 'WARD_NAME'
else:
    ward_name_col = None
    print("No ward name column found, using index as identifier")

# Create a DataFrame with ward information and counts
if ward_name_col:
    ward_info = delhi_gdf[[ward_name_col]].copy()
    ward_info['occurrences'] = region_counts.reindex(delhi_gdf.index, fill_value=0)
    
    # Sort by occurrences to find top 5
    top_5_wards = ward_info.sort_values('occurrences', ascending=False).head(5)
    print("\nTop 5 Wards by Number of Complaints:")
    print(top_5_wards)
else:
    # Use index if no name column is available
    ward_info = pd.DataFrame(index=delhi_gdf.index)
    ward_info['occurrences'] = region_counts.reindex(delhi_gdf.index, fill_value=0)
    
    # Sort by occurrences to find top 5
    top_5_wards = ward_info.sort_values('occurrences', ascending=False).head(5)
    print("\nTop 5 Wards by Number of Complaints (using index):")
    print(top_5_wards)

# Merge the counts back to the GeoDataFrame
delhi_gdf['occurrences'] = region_counts.reindex(delhi_gdf.index, fill_value=0)

# Reset the index to ensure 'index' is available as a column
delhi_gdf = delhi_gdf.reset_index()

# Print occurrences for validation
print("\nOccurrences in each zone (All Complaints):")
print(delhi_gdf[['index', 'occurrences']])

# Create Folium map (this will still create the HTML version)
m = folium.Map(location=[28.7041, 77.1025], zoom_start=11)

# Create a larger background rectangle (much bigger than Delhi's bounds)
min_x, min_y, max_x, max_y = delhi_gdf.total_bounds
padding = 1
background_box = box(
    min_x - padding,
    min_y - padding,
    max_x + padding,
    max_y + padding)

# Create the mask by subtracting Delhi's shape from the background
delhi_union = delhi_gdf.geometry.union_all()

mask = background_box.difference(delhi_union)

# Add the white background mask to the map
folium.GeoJson(mask.__geo_interface__, style_function=lambda x: {'fillColor': 'white', 'color': 'white', 'fillOpacity': 1, 'weight': 0}).add_to(m)

# Add the choropleth layer
folium.Choropleth(geo_data=delhi_gdf.__geo_interface__, data=delhi_gdf, columns=['index', 'occurrences'], key_on='feature.properties.index',
                  fill_color='YlOrRd', fill_opacity=0.7, line_opacity=0.2, legend_name='Number of All Occurrences').add_to(m)

# Save the map to an HTML file
m.save('delhi_all_complaints_map.html')

# Now create a static map with matplotlib for JPEG export
# Create a figure with white background
fig, ax = plt.subplots(figsize=(7, 7), facecolor='white')
ax.set_facecolor('white')

# Create a colormap similar to YlOrRd in Folium
colors_list = ['#ffffcc', '#ffeda0', '#fed976', '#feb24c', '#fd8d3c', '#fc4e2a', '#e31a1c', '#bd0026', '#800026']
cmap = colors.LinearSegmentedColormap.from_list('YlOrRd', colors_list)

# Plot the choropleth
delhi_gdf.plot(column='occurrences', cmap=cmap, linewidth=0.8, edgecolor='black', alpha=0.7, ax=ax)

# Remove axis
ax.set_axis_off()

# Get the current axis limits
x_min, x_max = ax.get_xlim()
y_min, y_max = ax.get_ylim()

# Add padding to ensure the white background covers everything
padding_factor = 0.1
width = x_max - x_min
height = y_max - y_min
x_min -= width * padding_factor
x_max += width * padding_factor
y_min -= height * padding_factor
y_max += height * padding_factor

# Reset the axis limits with padding
ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)

# Add title
plt.title('Delhi Open-Burning Complaints Map', fontsize=14)

# Create a custom legend
# Get the min and max values for the colorbar
vmin = delhi_gdf['occurrences'].min()
vmax = delhi_gdf['occurrences'].max()

# Create a colorbar separately
sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=vmin, vmax=vmax))
sm.set_array([])  # You need to set an array for the ScalarMappable
cbar = plt.colorbar(sm, ax=ax, orientation='horizontal', fraction=0.046, pad=0.04)
cbar.set_label('Number of All Complaints', fontsize=10)

# Optional: Highlight the top 5 wards on the map
# This requires finding the IDs after reset_index
if len(top_5_wards) > 0:
    # Get original indices of top 5 wards
    top_indices = top_5_wards.index.tolist()
    
    # Find these in the reset index dataframe
    for idx in top_indices:
        # Find the matching row in delhi_gdf
        ward_row = delhi_gdf[delhi_gdf['index'] == idx]
        if not ward_row.empty:
            # Add a thicker outline to highlight this ward
            ward_row.boundary.plot(ax=ax, color='black', linewidth=2.5)

# Save as JPEG with tight layout and white background
plt.savefig('delhi_all_complaints_map.jpg', 
            dpi=300, 
            bbox_inches='tight', 
            facecolor='white', 
            edgecolor='none')

print("\nMap saved as delhi_all_complaints_map.jpg")

# Show the map
plt.show()

In [ ]:
# 1. Load Excel File
file_path = r"C:\Users\Jeen priyee\Downloads\CEEW-Daily report_GDA-open-burning\complaint-1oct-31dec.xlsx" # CHANGE THIS ONLY
df = pd.read_excel(file_path)

# 2. Fix mojibake Hindi encoding
def fix_mojibake(x):
    if isinstance(x, str):
        try:
            return x.encode("latin1").decode("utf-8")
        except:
            return x
    return x

df = df.map(fix_mojibake)

# 3. Split Latitude & Longitude
df[['Latitude', 'Longitude']] = df['Latitude & Longitude'].str.split(",", expand=True)
df['Latitude'] = df['Latitude'].astype(float)
df['Longitude'] = df['Longitude'].astype(float)

# 4. Filter Burning Complaints (Offences 74 & 113)
burning_df = df[df["Offences"].isin([74,113])].copy()

print("Total Burning Complaints:", len(burning_df))

# 5. Load Delhi Shapefile (Wards)
shp_path = r"C:\Users\Jeen priyee\Downloads\Delhi_Wards-SHP_Datameet\Delhi_Wards.shp"
delhi_gdf = gpd.read_file(shp_path).to_crs(4326)

# 6. Create Folium Map (Delhi centered)
m = folium.Map(location=[28.7041, 77.1025], zoom_start=11, tiles="cartodbpositron")

# 7. Mask Outside Delhi — show only Delhi region
min_x, min_y, max_x, max_y = delhi_gdf.total_bounds
big_box = box(min_x - 1, min_y - 1, max_x + 1, max_y + 1)

delhi_union = delhi_gdf.union_all()
mask = big_box.difference(delhi_union)

# Add mask
folium.GeoJson(mask.__geo_interface__,
    style_function=lambda x: {"fillColor": "white", "color": "white", "fillOpacity": 1, "weight": 0}).add_to(m)

# Add Delhi outline
folium.GeoJson(delhi_gdf.__geo_interface__,
    style_function=lambda x: {"fillColor": "none", "color": "black", "weight": 1}).add_to(m)

# 8. Add Burning Complaint Markers With Color Based on Status
for _, row in burning_df.iterrows():

    # NEW LOGIC (Status → Marker Color)
    status = str(row["Status"]).lower().strip()
    if status == "pending":
        marker_color = "red"
    else:
        marker_color = "green"

    popup_html = f"""
    <b>Complaint ID:</b> {row['Compliant ID']}<br>
    <b>Address:</b> {row['Address']}<br>
    <b>Status:</b> {row['Status']}<br>
    <b>Offence:</b> {row['Offences']}
    """
    
    folium.Marker(
        [row["Latitude"], row["Longitude"]],
        popup=folium.Popup(popup_html, max_width=250),
        icon=folium.Icon(color=marker_color, icon="info-sign")
    ).add_to(m)

# 9. Save & Display in Jupyter Notebook
output_file = "Delhi_Burning_Complaints_Markers.html"
m.save(output_file)

print("\nSaved:", output_file)

# Display inside notebook
IFrame(output_file, width=900, height=600)


In [ ]:
import folium
import geopandas as gpd

# Load Delhi Shapefile
shp_path = r"C:\Users\Jeen priyee\Downloads\Delhi_Wards-SHP_Datameet\Delhi_Wards.shp"
delhi_gdf = gpd.read_file(shp_path)

# Convert CRS to WGS84 (required for folium)
delhi_gdf = delhi_gdf.to_crs(epsg=4326)

# Get boundary coordinates for map fitting
bounds = delhi_gdf.total_bounds  # [minx, miny, maxx, maxy]
minx, miny, maxx, maxy = bounds

# Create Folium Map centered on Delhi
m = folium.Map(location=[(miny + maxy)/2, (minx + maxx)/2], zoom_start=11)

# Add Delhi Boundary to Map
folium.GeoJson(delhi_gdf, name="Delhi Boundary",
    style_function=lambda x: {
        'fillColor': '#00000000',  # Transparent fill
        'color': 'blue',           # Boundary line
        'weight': 2}).add_to(m)

# Split 'Latitude & Longitude' into columns
df[['Latitude', 'Longitude']] = df['Latitude & Longitude'].str.split(",", expand=True)
df['Latitude'] = df['Latitude'].astype(float)
df['Longitude'] = df['Longitude'].astype(float)

# Add markers on the Delhi map
for _, row in burning_df.iterrows():
    color = 'red' if row['Status'].lower() == 'pending' else 'green'
    
    popup_text = f"Compliant ID: {row['Compliant ID']}\nAddress: {row['Address']}"
    
    folium.Marker(
        [row['Latitude'], row['Longitude']],popup=popup_text,
        icon=folium.Icon(color=color)).add_to(m)

# Fit map view to the shapefile boundary
m.fit_bounds([[miny, minx], [maxy, maxx]])

# Save the output
m.save("burning_map_delhi.html")
m


***11. Final Daily Report Output***

In [ ]:
report_text = f"""
1. TOTAL COMPLAINTS RECEIVED: {total_complaints}

2. TOTAL BURNING-RELATED COMPLAINTS (Offence 74 & 113): {burning_count}

3. DISTRICT-WISE DISTRIBUTION OF BURNING COMPLAINTS:
{district_burning.to_string()}


4. DETAILS OF ALL BURNING COMPLAINTS:
{burning_details.to_string(index=False)}

Map file generated: burning_map.html
"""

print(report_text)


In [ ]:
#Code for GVP identification
import pandas as pd
import folium
import geopandas as gpd
from sklearn.cluster import DBSCAN
import numpy as np
import matplotlib.colors as mcolors
import matplotlib.cm as cm
from tabulate import tabulate
from matplotlib import colormaps


# Load the CSV file with a specified encoding
complaints_df = burning_df.copy()

# Remove rows 2 to 57556
complaints_df['Date and Time'] = pd.to_datetime(complaints_df['Date and Time'])
complaints_df = complaints_df[complaints_df['Date and Time'] >= '2024-01-01']


# Split latitude and longitude into separate columns
complaints_df[['Latitude', 'Longitude']] = complaints_df['Latitude & Longitude'].str.split(',', expand=True)
complaints_df['Latitude'] = complaints_df['Latitude'].astype(float)
complaints_df['Longitude'] = complaints_df['Longitude'].astype(float)

# Filter for "Dumping of Construction & Demolition Waste" offences
open_burning_df = complaints_df.copy()

# Convert coordinates to radians for haversine calculation
open_burning_df['Latitude_rad'] = np.radians(open_burning_df['Latitude'])
open_burning_df['Longitude_rad'] = np.radians(open_burning_df['Longitude'])

# Apply DBSCAN with a 5-meter radius (approximately 0.000045 radians) and minimum samples of 1
coords = open_burning_df[['Latitude_rad', 'Longitude_rad']].values
db = DBSCAN(eps=0.00000085, min_samples=1, metric='haversine').fit(coords)

# Add cluster labels to the DataFrame
open_burning_df['Cluster'] = db.labels_

# Group by clusters and count occurrences
clustered_counts = open_burning_df.groupby('Cluster').size().reset_index(name='Occurrences')

# Get the mean location for each cluster
cluster_centers = open_burning_df.groupby('Cluster').agg({
    'Latitude': 'mean',
    'Longitude': 'mean',
    'Geo Location': 'first',  # Take the first occurrence of Geo Location as representative
}).reset_index()

# Merge the counts with the cluster centers
clustered_data = pd.merge(cluster_centers, clustered_counts, on='Cluster')

# Add a column to identify if the cluster has any pending status complaints
pending_status_by_cluster = open_burning_df.groupby('Cluster')['Status'].apply(
    lambda x: 'Pending' in x.values).reset_index(name='Has_Pending')

# Merge this information with clustered_data
clustered_data = pd.merge(clustered_data, pending_status_by_cluster, on='Cluster')

# Filter for clusters that have pending status
pending_clusters = clustered_data[clustered_data['Has_Pending']].copy()

# Tabulate top 100 occurrences with pending status
top_100_occurrences = pending_clusters.sort_values(by='Occurrences', ascending=False).head(100)

# Normalize occurrences for colormap
norm = mcolors.Normalize(vmin=top_100_occurrences['Occurrences'].min(), vmax=top_100_occurrences['Occurrences'].max())
cmap = cm.get_cmap('YlOrRd')


# Load the shapefile
shapefile_path = r'C:\Users\Jeen priyee\Downloads\Delhi_Wards-SHP_Datameet\Delhi_Wards.shp'

wards_gdf = gpd.read_file(shapefile_path)

# Set CRS to EPSG:4326 if not already set
if wards_gdf.crs is None:
    wards_gdf.set_crs(epsg=4326, inplace=True)
else:
    wards_gdf.to_crs(epsg=4326, inplace=True)

# Create a folium map with a white background
m = folium.Map(
    location=[28.7041, 77.1025], 
    zoom_start=11,
    tiles="cartodbpositron",  # Use a white base map
    attr="CartoDB Positron"
)

# Create a white overlay for entire map area first
folium.Rectangle(
    bounds=[[wards_gdf.total_bounds[1], wards_gdf.total_bounds[0]], 
            [wards_gdf.total_bounds[3], wards_gdf.total_bounds[2]]],
    color='white',
    fill=True,
    fill_color='white',
    fill_opacity=1.0
).add_to(m)

# Add ward boundaries to the map
folium.GeoJson(
    wards_gdf,
    name='Delhi Wards',
    style_function=lambda x: {'fillColor': 'transparent', 'color': 'black', 'weight': 1}
).add_to(m)

# Add circle markers for each of the top 100 clusters with occurrences count
for idx, row in top_100_occurrences.iterrows():
    cluster_complaints = open_burning_df[open_burning_df['Cluster'] == row['Cluster']]
    
    # Get pending complaints in this cluster
    pending_complaints = cluster_complaints[cluster_complaints['Status'] == 'Pending']

    dates_str = cluster_complaints['Date and Time'].dropna().dt.strftime('%d-%b-%Y').unique()

    resolve_images = ''.join([f"<img src='{img}' width='150' height='150'><br>" for img in cluster_complaints['Resolve Image'].dropna()])
    offence_images = ''.join([f"<img src='{img}' width='150' height='150'><br>" for img in cluster_complaints['Offence Image'].dropna()])
    color = mcolors.to_hex(cmap(norm(row['Occurrences'])))
    
    # Add pending status count to popup
    pending_count = len(pending_complaints)
    total_count = len(cluster_complaints)
    
    popup_content = (f"Occurrences: {row['Occurrences']}<br>"
                     f"Pending Complaints: {pending_count} of {total_count}<br>"
                     f"Location: {row['Geo Location']}<br>"
                     f"Date(s): {', '.join(dates_str)}<br>"
                     f"Status: {', '.join(cluster_complaints['Status'].dropna())}<br>"
                     f"Resolve Images:<br>{resolve_images}<br>"
                     f"Offence Images:<br>{offence_images}")
    
    folium.CircleMarker(
        location=[row['Latitude'], row['Longitude']],
        radius=5 + row['Occurrences'] / 10,  # Increase radius based on occurrences
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.7,
        popup=folium.Popup(popup_content, max_width=300, max_height=600)
    ).add_to(m)

# Add legend
legend_html = '''
<div style="position: fixed; 
    bottom: 50px; left: 50px; width: 250px; height: 150px; 
    border:2px solid grey; z-index:9999; font-size:14px;
    background-color:white;
    padding: 10px;
    ">
    <b>Occurrences Legend</b><br>
    <i style="background:#ffffb2; width: 20px; height: 20px; display: inline-block;"></i>&nbsp;Low (1-3)<br>
    <i style="background:#fd8d3c; width: 20px; height: 20px; display: inline-block;"></i>&nbsp;Medium (4-10)<br>
    <i style="background:#bd0026; width: 20px; height: 20px; display: inline-block;"></i>&nbsp;High (>10)<br>
    <br>
    <i>Note: Only showing locations with pending complaints</i>
</div>
'''

m.get_root().html.add_child(folium.Element(legend_html))

# Remove the folium attribution and any other map elements except what we want
m.get_root().html.add_child(folium.Element('''
<style>
.leaflet-control-attribution {
    display: none;
}
.leaflet-control-zoom {
    display: none;
}
</style>
'''))

# Save the map to an HTML file to view it in a browser
m.save('delhi_open_burning_complaints_map.html')

# Display the top 10 locations and their occurrence numbers
top_10_locations = top_100_occurrences[['Occurrences', 'Geo Location', 'Has_Pending']].head(10)

# Enhanced tabulation output for top 10 locations
print("\nTop 10 Locations with Pending Complaints:")
print(tabulate(top_10_locations[['Occurrences', 'Geo Location']], headers=['Occurrences', 'Geo Location'], tablefmt='fancy_grid', showindex=False, stralign='center'))

# Display the map in the notebook (if supported)
m

In [ ]:
hotspot_summary = (
    open_burning_df
    .groupby('Cluster')
    .agg(
        complaints=('Cluster', 'count'),
        lat_mean=('Latitude', 'mean'),
        lon_mean=('Longitude', 'mean'),
        Location=('Geo Location', 'first')
    )
    .reset_index()
)

hotspot_summary = hotspot_summary.sort_values(
    by='complaints', ascending=False
).reset_index(drop=True)

hotspot_summary.insert(
    0,
    'Hotspot',
    ['Hotspot ' + str(i+1) for i in range(len(hotspot_summary))]
)

hotspot_summary = hotspot_summary[hotspot_summary['complaints'] >= 2]

hotspot_summary = hotspot_summary[
    ['Hotspot', 'Location', 'lat_mean', 'lon_mean', 'Ward_Name', 'complaints']
]

output_file = 'cluster_summary.xlsx'
hotspot_summary.to_excel(output_file, index=False)

print(f"Hotspot summary saved as {output_file}")


In [ ]:
import pandas as pd
import folium
import geopandas as gpd
from sklearn.cluster import DBSCAN
import numpy as np
import matplotlib.colors as mcolors
import matplotlib.cm as cm
from tabulate import tabulate

# ------------------------------------
# 1. DATA PREPARATION
# ------------------------------------

complaints_df = burning_df.copy()

complaints_df['Date and Time'] = pd.to_datetime(
    complaints_df['Date and Time'], errors='coerce'
)
complaints_df = complaints_df[
    complaints_df['Date and Time'] >= '2024-01-01'
]

complaints_df[['Latitude', 'Longitude']] = (
    complaints_df['Latitude & Longitude']
    .str.split(',', expand=True)
)

complaints_df['Latitude'] = complaints_df['Latitude'].astype(float)
complaints_df['Longitude'] = complaints_df['Longitude'].astype(float)

# ------------------------------------
# 2. DBSCAN CLUSTERING (5 METERS)
# ------------------------------------

complaints_df['Latitude_rad'] = np.radians(complaints_df['Latitude'])
complaints_df['Longitude_rad'] = np.radians(complaints_df['Longitude'])

coords = complaints_df[['Latitude_rad', 'Longitude_rad']].values

earth_radius = 6371000  # meters
eps_radians = 5 / earth_radius

db = DBSCAN(
    eps=eps_radians,
    min_samples=1,
    metric='haversine'
).fit(coords)

complaints_df['Cluster'] = db.labels_

# ------------------------------------
# 3. CLUSTER SUMMARY (ALL COMPLAINTS)
# ------------------------------------

cluster_summary = (
    complaints_df
    .groupby('Cluster')
    .agg(
        complaints=('Cluster', 'count'),      # ✅ ALL complaints
        lat_mean=('Latitude', 'mean'),
        lon_mean=('Longitude', 'mean'),
        Location=('Geo Location', 'first'),
        pending_count=('Status', lambda x: (x == 'Pending').sum())
    )
    .reset_index()
)

# ✅ Hotspot condition: ≥ 2 complaints
cluster_summary = cluster_summary[
    cluster_summary['complaints'] >= 2
].copy()

# ------------------------------------
# 4. ADD WARD NAME FROM SHP
# ------------------------------------

shapefile_path = r'C:\Users\Jeen priyee\Downloads\Delhi_Wards-SHP_Datameet\Delhi_Wards.shp'
wards_gdf = gpd.read_file(shapefile_path).to_crs(epsg=4326)

cluster_gdf = gpd.GeoDataFrame(
    cluster_summary,
    geometry=gpd.points_from_xy(
        cluster_summary['lon_mean'],
        cluster_summary['lat_mean']
    ),
    crs='EPSG:4326'
)

cluster_gdf = gpd.sjoin(
    cluster_gdf,
    wards_gdf[['Ward_Name', 'geometry']],
    how='left',
    predicate='within'
)

# ------------------------------------
# 5. TOP 100 HOTSPOTS
# ------------------------------------

top_100 = (
    cluster_gdf
    .sort_values(by='complaints', ascending=False)
    .head(100)
    .reset_index(drop=True)
)

# ------------------------------------
# 6. MAP CREATION
# ------------------------------------

norm = mcolors.Normalize(
    vmin=top_100['complaints'].min(),
    vmax=top_100['complaints'].max()
)
cmap = cm.get_cmap('YlOrRd')

m = folium.Map(
    location=[28.7041, 77.1025],
    zoom_start=11,
    tiles="cartodbpositron"
)

folium.GeoJson(
    wards_gdf,
    style_function=lambda x: {
        'fillColor': 'transparent',
        'color': 'black',
        'weight': 1
    }
).add_to(m)

for _, row in top_100.iterrows():

    popup_content = f"""
    <b>Total Complaints:</b> {row['complaints']}<br>
    <b>Pending Complaints:</b> {row['pending_count']}<br>
    <b>Location:</b> {row['Location']}<br>
    <b>Ward:</b> {row['Ward_Name']}
    """

    color = mcolors.to_hex(cmap(norm(row['complaints'])))

    folium.CircleMarker(
        location=[row['lat_mean'], row['lon_mean']],
        radius=5 + row['complaints'] / 10,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.7,
        popup=folium.Popup(popup_content, max_width=300)
    ).add_to(m)

m.save("hotspots_all_complaints.html")

# ------------------------------------
# 7. EXCEL OUTPUT (SAME AS YOUR SAMPLE)
# ------------------------------------

top_100_excel = top_100.copy()

top_100_excel.insert(
    0,
    'Hotspot',
    ['Hotspot ' + str(i + 1) for i in range(len(top_100_excel))]
)

top_100_excel = top_100_excel[
    ['Hotspot', 'Location', 'lat_mean', 'lon_mean', 'Ward_Name', 'complaints']
]

top_100_excel.to_excel("cluster_summary.xlsx", index=False)

print("Excel saved: cluster_summary.xlsx")

# ------------------------------------
# 8. DISPLAY TABLE (EXACT FORMAT)
# ------------------------------------

print("\nTop 10 Hotspots:")
print(
    tabulate(
        top_100_excel.head(20),
        headers='keys',
        tablefmt='fancy_grid',
        showindex=False,
        stralign='center'
    )
)

m


import os
from IPython.display import FileLink, display

download_path = os.path.join(
    os.path.expanduser("~"),
    "Downloads",
    "cluster_summary.xlsx"
)

top_100_excel.to_excel(download_path, index=False)

print(f"Excel saved to: {download_path}")
display(FileLink(download_path))


In [ ]:
import pandas as pd
import folium
import geopandas as gpd
from sklearn.cluster import DBSCAN
import numpy as np
import matplotlib.colors as mcolors
import matplotlib.cm as cm
from tabulate import tabulate
import os
from IPython.display import FileLink, display

# ------------------------------------
# 1. DATA PREPARATION
# ------------------------------------

complaints_df = burning_df.copy()

complaints_df['Date and Time'] = pd.to_datetime(
    complaints_df['Date and Time'], errors='coerce'
)
complaints_df = complaints_df[
    complaints_df['Date and Time'] >= '2024-01-01'
]

complaints_df[['Latitude', 'Longitude']] = (
    complaints_df['Latitude & Longitude']
    .str.split(',', expand=True)
)

complaints_df['Latitude'] = complaints_df['Latitude'].astype(float)
complaints_df['Longitude'] = complaints_df['Longitude'].astype(float)

# ------------------------------------
# 2. DBSCAN CLUSTERING (5 METERS)
# ------------------------------------

complaints_df['Latitude_rad'] = np.radians(complaints_df['Latitude'])
complaints_df['Longitude_rad'] = np.radians(complaints_df['Longitude'])

coords = complaints_df[['Latitude_rad', 'Longitude_rad']].values

earth_radius = 6371000
eps_radians = 5 / earth_radius

db = DBSCAN(
    eps=eps_radians,
    min_samples=1,
    metric='haversine'
).fit(coords)

complaints_df['Cluster'] = db.labels_

# ------------------------------------
# 3. CLUSTER SUMMARY (ALL COMPLAINTS)
# ------------------------------------

cluster_summary = (
    complaints_df
    .groupby('Cluster')
    .agg(
        complaints=('Cluster', 'count'),
        lat_mean=('Latitude', 'mean'),
        lon_mean=('Longitude', 'mean'),
        Location=('Geo Location', 'first'),
        pending_count=('Status', lambda x: (x == 'Pending').sum())
    )
    .reset_index()
)

# Hotspot rule: ≥ 2 complaints
cluster_summary = cluster_summary[
    cluster_summary['complaints'] >= 2
].copy()

# ------------------------------------
# 4. ADD WARD NAME FROM SHP
# ------------------------------------

shapefile_path = r'C:\Users\Jeen priyee\Downloads\Delhi_Wards-SHP_Datameet\Delhi_Wards.shp'
wards_gdf = gpd.read_file(shapefile_path).to_crs(epsg=4326)

cluster_gdf = gpd.GeoDataFrame(
    cluster_summary,
    geometry=gpd.points_from_xy(
        cluster_summary['lon_mean'],
        cluster_summary['lat_mean']
    ),
    crs='EPSG:4326'
)

cluster_gdf = gpd.sjoin(
    cluster_gdf,
    wards_gdf[['Ward_Name', 'geometry']],
    how='left',
    predicate='within'
)

# ------------------------------------
# 5. ONE HOTSPOT PER WARD (MAX COMPLAINTS)
# ------------------------------------

cluster_gdf = cluster_gdf.sort_values(
    by=['Ward_Name', 'complaints'],
    ascending=[True, False]
)

wardwise_hotspots = (
    cluster_gdf
    .drop_duplicates(subset='Ward_Name', keep='first')
    .reset_index(drop=True)
)

# ------------------------------------
# 6. MAP
# ------------------------------------

norm = mcolors.Normalize(
    vmin=wardwise_hotspots['complaints'].min(),
    vmax=wardwise_hotspots['complaints'].max()
)
cmap = cm.get_cmap('YlOrRd')

m = folium.Map(
    location=[28.7041, 77.1025],
    zoom_start=11,
    tiles="cartodbpositron"
)

folium.GeoJson(
    wards_gdf,
    style_function=lambda x: {
        'fillColor': 'transparent',
        'color': 'black',
        'weight': 1
    }
).add_to(m)

for _, row in wardwise_hotspots.iterrows():

    popup = f"""
    <b>Ward:</b> {row['Ward_Name']}<br>
    <b>Total Complaints:</b> {row['complaints']}<br>
    <b>Pending:</b> {row['pending_count']}<br>
    <b>Location:</b> {row['Location']}
    """

    color = mcolors.to_hex(cmap(norm(row['complaints'])))

    folium.CircleMarker(
        location=[row['lat_mean'], row['lon_mean']],
        radius=5 + row['complaints'] / 10,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.7,
        popup=popup
    ).add_to(m)

# ------------------------------------
# 7. EXCEL OUTPUT (WARD-WISE)
# ------------------------------------

wardwise_excel = wardwise_hotspots.copy()

wardwise_excel.insert(
    0,
    'Hotspot',
    ['Hotspot ' + str(i+1) for i in range(len(wardwise_excel))]
)

wardwise_excel = wardwise_excel[
    ['Hotspot', 'Ward_Name', 'Location',
     'lat_mean', 'lon_mean', 'complaints']
]

download_path = os.path.join(
    os.path.expanduser("~"),
    "Downloads",
    "wardwise_max_hotspots.xlsx"
)

wardwise_excel.to_excel(download_path, index=False)

print(f"Excel saved to: {download_path}")
display(FileLink(download_path))

# ------------------------------------
# 8. DISPLAY TABLE
# ------------------------------------

print("\nWard-wise Max Complaint Hotspots:")
print(
    tabulate(
        wardwise_excel.head(20),
        headers='keys',
        tablefmt='fancy_grid',
        showindex=False,
        stralign='center'
    )
)

m


In [ ]:
import pandas as pd
import folium
import geopandas as gpd
from sklearn.cluster import DBSCAN
import numpy as np
import matplotlib.colors as mcolors
import matplotlib.cm as cm
from tabulate import tabulate

# ------------------------------------
# 1. DATA PREPARATION (ALL YEARS)
# ------------------------------------

complaints_df = burning_df.copy()

complaints_df['Date and Time'] = pd.to_datetime(
    complaints_df['Date and Time'], errors='coerce'
)

# Split Latitude & Longitude
complaints_df[['Latitude', 'Longitude']] = (
    complaints_df['Latitude & Longitude']
    .str.split(',', expand=True)
)

complaints_df['Latitude'] = complaints_df['Latitude'].astype(float)
complaints_df['Longitude'] = complaints_df['Longitude'].astype(float)

# ------------------------------------
# 2. DBSCAN CLUSTERING (5 METERS)
# ------------------------------------

complaints_df['Latitude_rad'] = np.radians(complaints_df['Latitude'])
complaints_df['Longitude_rad'] = np.radians(complaints_df['Longitude'])

coords = complaints_df[['Latitude_rad', 'Longitude_rad']].values

earth_radius = 6371000  # meters
eps_meters = 5
eps_radians = eps_meters / earth_radius

db = DBSCAN(
    eps=eps_radians,
    min_samples=1,
    metric='haversine'
).fit(coords)

complaints_df['Cluster'] = db.labels_

# ------------------------------------
# 3. CLUSTER-LEVEL FILTERING
#    ≥2 complaints AND ≥1 Pending
# ------------------------------------

cluster_counts = (
    complaints_df
    .groupby('Cluster')
    .size()
    .reset_index(name='complaints')
)

cluster_pending = (
    complaints_df
    .groupby('Cluster')['Status']
    .apply(lambda x: 'Pending' in x.values)
    .reset_index(name='Has_Pending')
)

cluster_centers = (
    complaints_df
    .groupby('Cluster')
    .agg(
        lat_mean=('Latitude', 'mean'),
        lon_mean=('Longitude', 'mean'),
        Location=('Geo Location', 'first')
    )
    .reset_index()
)

cluster_summary = (
    cluster_centers
    .merge(cluster_counts, on='Cluster')
    .merge(cluster_pending, on='Cluster')
)

valid_clusters = cluster_summary[
    (cluster_summary['complaints'] >= 2) &
    (cluster_summary['Has_Pending'])
].copy()

valid_df = complaints_df[
    complaints_df['Cluster'].isin(valid_clusters['Cluster'])
].copy()

# ------------------------------------
# 4. SPATIAL JOIN → WARD NAME
# ------------------------------------

shapefile_path = r'C:\Users\Jeen priyee\Downloads\Delhi_Wards-SHP_Datameet\Delhi_Wards.shp'
wards_gdf = gpd.read_file(shapefile_path).to_crs(epsg=4326)

points_gdf = gpd.GeoDataFrame(
    valid_df,
    geometry=gpd.points_from_xy(valid_df.Longitude, valid_df.Latitude),
    crs='EPSG:4326'
)

points_with_ward = gpd.sjoin(
    points_gdf,
    wards_gdf[['Ward_Name', 'geometry']],
    how='left',
    predicate='within'
)

# ------------------------------------
# 5. ONE CLUSTER PER WARD (MAX COMPLAINTS)
# ------------------------------------

ward_cluster_summary = (
    points_with_ward
    .groupby(['Ward_Name', 'Cluster'])
    .agg(
        complaints=('Cluster', 'count'),
        lat_mean=('Latitude', 'mean'),
        lon_mean=('Longitude', 'mean'),
        Location=('Geo Location', 'first')
    )
    .reset_index()
)

wardwise_hotspots = (
    ward_cluster_summary
    .sort_values(['Ward_Name', 'complaints'], ascending=[True, False])
    .groupby('Ward_Name')
    .head(1)
)

# ------------------------------------
# 6. SORT DESCENDING (MAX ON TOP)
# ------------------------------------

wardwise_hotspots = (
    wardwise_hotspots
    .sort_values(by='complaints', ascending=False)
    .reset_index(drop=True)
)

# ------------------------------------
# 7. MAP (OPTIONAL BUT MATCHES OUTPUT)
# ------------------------------------

norm = mcolors.Normalize(
    vmin=wardwise_hotspots['complaints'].min(),
    vmax=wardwise_hotspots['complaints'].max()
)
cmap = cm.get_cmap('YlOrRd')

m = folium.Map(
    location=[28.7041, 77.1025],
    zoom_start=11,
    tiles='cartodbpositron'
)

folium.GeoJson(
    wards_gdf,
    style_function=lambda x: {
        'fillColor': 'transparent',
        'color': 'black',
        'weight': 1
    }
).add_to(m)

for _, row in wardwise_hotspots.iterrows():

    color = mcolors.to_hex(cmap(norm(row['complaints'])))

    popup = f"""
    <b>Ward:</b> {row['Ward_Name']}<br>
    <b>Total Complaints:</b> {row['complaints']}<br>
    <b>Location:</b> {row['Location']}
    """

    folium.CircleMarker(
        location=[row['lat_mean'], row['lon_mean']],
        radius=6 + row['complaints'] / 10,
        color=color,
        fill=True,
        fill_opacity=0.7,
        popup=popup
    ).add_to(m)

m.save('wardwise_max_hotspots.html')

# ------------------------------------
# 8. EXCEL OUTPUT (DOWNLOADABLE)
# ------------------------------------

excel_df = wardwise_hotspots.copy()

excel_df.insert(
    0,
    'Hotspot',
    ['Hotspot ' + str(i + 1) for i in range(len(excel_df))]
)

excel_df = excel_df[
    ['Hotspot', 'Ward_Name', 'Location', 'lat_mean', 'lon_mean', 'complaints']
]

excel_path = 'wardwise_max_hotspots.xlsx'
excel_df.to_excel(excel_path, index=False)

print(f"Excel saved: {excel_path}")

# ------------------------------------
# 9. TOP 10 TABLE (DISPLAY)
# ------------------------------------

print("\nTop 10 Ward-wise Hotspots:")
print(
    tabulate(
        excel_df.head(20),
        headers='keys',
        tablefmt='fancy_grid',
        showindex=False
    )
)

m

excel_path = r"C:\Users\Jeen priyee\Desktop\wardwise_max_indexed_hotspots.xlsx"
excel_df.to_excel(excel_path, index=False)

print("Excel saved on Desktop")
